# TP : Proximal Policy Optimization (PPO)

Dans ce TP, nous implémentons **PPO** sur l'environnement CartPole. Contrairement à DQN, qui apprend une Q-value et en déduit une politique (greedy), PPO apprend **directement une politique stochastique** $\pi_\theta(a|s)$, aidée d'une fonction de valeur $V_\phi(s)$ : c'est un algorithme de type **acteur-critique**.

# Imports

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

# Partie 1 : Outils

Cette classe permet d'agréger les frames des env.render() puis d'afficher l'épisode à la fin.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

class EpisodeAnimator:
    def __init__(self, interval: int = 200):
        """
        interval: delay between frames in milliseconds
        """
        self.frames = []
        self.interval = interval
    
    def add_frame(self, frame):
        """Add a single frame (numpy RGB array) to the animation."""
        self.frames.append(frame)
    
    def display_animation(self):
        """Render and display the animation inline in a Jupyter notebook."""
        if not self.frames:
            raise ValueError("No frames to animate. Use add_frame() first.")
        
        fig = plt.figure()
        img = plt.imshow(self.frames[0])
        plt.axis('off')

        def animate(i):
            img.set_data(self.frames[i])
            return [img]
        
        ani = animation.FuncAnimation(
            fig, animate, frames=len(self.frames),
            interval=self.interval, blit=True
        )
        plt.close(fig)  # Prevent static last frame from showing
        return HTML(ani.to_jshtml())

Cette fonction joue (évalue) une certaine politique dans un certain environnement précisés pendant 1 épisode, et retourne les récompenses cumulées ainsi que les frames de l'épisode.

In [ ]:
from typing import Callable


def eval_policy_1_episode(env : gym.Env, policy : Callable, do_animation : bool = True):
    observation, info = env.reset()
    done = False
    total_reward = 0
    step = 0
    animator = EpisodeAnimator(interval=int(1000 / env.metadata["render_fps"])) if do_animation else None  # real-time speed

    while not done:

        if do_animation:
            animator.add_frame(env.render())

        action = policy(observation)
        observation, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        total_reward += reward
        step += 1

    if do_animation:
        animator.add_frame(env.render()) # add last frame
        animation = animator.display_animation()
    else:
        animation = None

    return {"total_reward": total_reward, "steps": step, "animation" : animation}

Cette fonction évalue une politique sur plusieurs épisodes (seul le dernier est animé) :

In [ ]:
def eval_policy_n_episodes(env, policy, n=5, do_animation=True):
    """Evaluate a policy over multiple episodes.

    Args:
        env (gym.Env): The environment to evaluate the policy on.
        policy (Callable): the policy to evaluate : policy(observation) -> action
        n (int, optional): The number of episodes to evaluate the policy on. Defaults to 5.
        do_animation (bool, optional): Whether to render the environment during evaluation. Only the last episode is rendered. Defaults to True.

    Returns:
        dict: A dictionary containing the evaluation results.
    """
    results = {}
    for i in tqdm(range(n)):
        result_episode = eval_policy_1_episode(env, policy, do_animation=do_animation and i == n-1)
        for key, value in result_episode.items():
            if key not in results:
                results[key] = []
            results[key].append(value)
    return results

## Framework for Reinforcement Learning Agents

Pour faire les choses proprement nous allons définir une classe de base pour les agents de RL.
Cette classe définit une interface python à respecter pour tout agent de RL. Les méthodes suivantes doivent être implémentées :

- `__init__(self, env: gym.Env, **kwargs)`: Le constructeur prend un environnement Gymnasium comme entrée (et d'autres paramètres dépendant de l'agent si besoin)
- `action = act(self, obs)`: Cette méthode doit retourner l'action à entreprendre en fonction de l'observation de l'état actuel.
- `learn(self, obs, action, reward, next_obs, terminated, truncated)`: Cette méthode doit mettre à jour la politique de l'agent en fonction de l'expérience acquise sur la dernière transition. Comme dans `env.step()`, il y a deux façons pour un épisode de s'arrêter :
    - `terminated` : l'épisode s'est terminé dans un état terminal (ici : le bâton est tombé). Il n'y a plus de récompense future après `next_obs`.
    - `truncated` : l'épisode a été coupé par une limite de temps. `next_obs` n'est **pas** un état terminal : l'agent aurait pu continuer à recevoir des récompenses.

In [ ]:
class AgentRL:

    def __init__(self, env : gym.Env):
        self.env = env

    def act(self, obs):
        raise NotImplementedError()

    def learn(self, obs, action, reward, next_obs, terminated, truncated):
        raise NotImplementedError()

Un exemple d'agent (aléatoire) respectant ce framework :

In [ ]:
class RandomAgent(AgentRL):

    def act(self, obs):
        return self.env.action_space.sample()

    def learn(self, obs, action, reward, next_obs, terminated, truncated):
        pass  # do nothing

Une fois votre agent implémenté, ces deux fonctions serviront respectivement à l'entraîner et à l'évaluer.

In [ ]:
def train_agent_n_episodes(env : gym.Env, agent : AgentRL, n : int = 20, verbose_freq : int = None) -> dict:
    """Train the agent for a number of episodes.

    Args:
        env (gym.Env): The environment to train the agent in.
        agent (AgentRL): The agent to train.
        n (int, optional): The number of episodes to train the agent for. Defaults to 20.
        verbose_freq (int, optional): Episode frequency to print progress. If None, no progress is printed. Defaults to None.

    Returns:
        dict: A dictionary containing the total rewards and steps taken in each episode.
    """
    results = {
        "total_rewards": [],
        "steps": [],
    }

    # Training loop
    for episode in tqdm(range(n)):
        observation, info = env.reset()
        total_reward = 0
        step = 0
        done = False
        while not done:
            action = agent.act(observation)
            next_observation, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            total_reward += reward
            agent.learn(observation, action, reward, next_observation, terminated, truncated)
            step += 1
            observation = next_observation

        # Log results
        results["total_rewards"].append(total_reward)
        results["steps"].append(step)
        if verbose_freq is not None and (episode + 1) % verbose_freq == 0:
            avg_reward = np.mean(results["total_rewards"][-verbose_freq:])
            avg_steps = np.mean(results["steps"][-verbose_freq:])
            print(f"Episode {episode + 1}/{n} - Average Reward: {avg_reward:.2f}, Average Steps: {avg_steps:.2f}")

    return results

def eval_agent_n_episodes(env : gym.Env, agent : AgentRL, n : int = 5, do_animation : bool = True) -> dict:
    return eval_policy_n_episodes(env, agent.act, n=n, do_animation=do_animation)

Cette fonction affiche les courbes d'apprentissage renvoyées par `train_agent_n_episodes` :

In [ ]:
import matplotlib.pyplot as plt

def plot_results(results: dict):
    """
    Plots each metric in results on a separate subplot.
    Example results:
        {
            'total_rewards': [0.0, 1.0, 0.0],
            'steps': [100, 95, 100]
        }
    """
    n_metrics = len(results)
    fig, axes = plt.subplots(n_metrics, 1, figsize=(10, 4 * n_metrics), sharex=True)

    if n_metrics == 1:
        axes = [axes]  # Make iterable if only one metric

    for ax, (key, values) in zip(axes, results.items()):
        ax.plot(values, label=key)
        ax.set_title(key.replace("_", " ").title())
        ax.set_ylabel("Value")
        ax.grid(True)
        ax.legend()

    axes[-1].set_xlabel("Episode")
    plt.tight_layout()
    plt.show()

# Partie 2 : L'environnement CartPole

Nous allons travailler sur l'environnement [CartPole](https://gymnasium.farama.org/environments/classic_control/cart_pole/). Il s'agit de maintenir un bâton en équilibre sur un chariot en poussant ce dernier vers la gauche ou vers la droite.

- **Observation** : un vecteur de 4 nombres réels (position du chariot, vitesse du chariot, angle du bâton, vitesse angulaire du bâton).
- **Actions** : 0 (pousser à gauche) ou 1 (pousser à droite).
- **Récompense** : +1 à chaque pas de temps où le bâton tient debout. L'épisode se termine (`terminated`) lorsque le bâton tombe (angle > 12°) ou que le chariot sort de l'écran, et il est tronqué (`truncated`) après 200 pas : le score maximal est donc 200.

L'espace d'observation est continu : la politique et la fonction de valeur seront représentées par des réseaux de neurones.

L'objet `env` est un environnement Gym. Ses méthodes principales sont:
- `obs, info = env.reset()` : réinitialise l'environnement et renvoie l'état initial (observation) et des informations supplémentaires.
- `obs, reward, terminated, truncated, info = env.step(action)` : effectue l'action `action` dans l'environnement et renvoie le nouvel état (observation), la reward, un booléen indiquant si l'épisode est terminé, un booléen indiquant si l'épisode a été tronqué et des informations supplémentaires.
- `img = env.render()` : renvoie une image de l'état actuel de l'environnement.

In [ ]:
env_cartpole = gym.make("CartPole-v1", max_episode_steps=200, render_mode="rgb_array")
print("Observation space:", env_cartpole.observation_space)
print("Action space:", env_cartpole.action_space)
obs, info = env_cartpole.reset()
print("Observation:", obs)
plt.imshow(env_cartpole.render())
plt.axis('off')
plt.show()

Pour avoir un point de comparaison, regardons les performances d'un agent aléatoire :

In [ ]:
# Trying a random agent in CartPole
agent = RandomAgent(env_cartpole)

# Evaluate the agent
eval_results = eval_agent_n_episodes(env_cartpole, agent, n=10)
print(f"Total reward: {eval_results['total_reward']}, mean = {np.mean(eval_results['total_reward'])}")
print(f"Total steps: {eval_results['steps']}, mean = {np.mean(eval_results['steps'])}")
print(f"Animation (last episode):")
eval_results['animation'][-1]

# Partie 3 : Proximal Policy Optimization (PPO)

## A) Principe de PPO

PPO alterne deux phases :

1. **Collecte (rollout)** : l'agent joue `n_steps` pas de temps avec sa politique actuelle $\pi_{\theta_{old}}$ et stocke les transitions, ainsi que $\log \pi_{\theta_{old}}(a_t|s_t)$, la log-probabilité de chaque action jouée.
2. **Mise à jour** : on calcule les avantages de chaque transition, puis on fait plusieurs passes (`n_epochs`) de descente de gradient par mini-batchs sur ces données, et on vide le buffer.

PPO est un algorithme **on-policy** : les données doivent avoir été collectées par la politique actuelle. Contrairement au replay buffer de DQN, qui garde les transitions de nombreuses politiques passées, le **rollout buffer** de PPO est vidé après chaque mise à jour. PPO peut tout de même réutiliser un même rollout pendant plusieurs epochs, grâce au "clipping" décrit ci-dessous, qui empêche la politique de trop s'éloigner de celle qui a collecté les données.

**Avantage (GAE).** L'avantage $\hat{A}_t$ mesure à quel point l'action $a_t$ a été meilleure que prévu par le critique. On utilise le Generalized Advantage Estimation :
$$\delta_t = r_t + \gamma V_\phi(s_{t+1}) (1 - \text{terminated}_t) - V_\phi(s_t)$$
$$\hat{A}_t = \delta_t + \gamma \lambda (1 - \text{done}_t) \hat{A}_{t+1}$$
calculé de la fin du rollout vers le début, avec $\text{done}_t = \text{terminated}_t$ ou $\text{truncated}_t$ (on ne propage pas l'avantage d'un épisode à l'autre). La cible de la fonction de valeur est alors $R_t = \hat{A}_t + V_\phi(s_t)$.

**Objectif "clippé" de l'acteur.** Avec le ratio $\rho_t(\theta) = \dfrac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)} = \exp\left(\log \pi_\theta(a_t|s_t) - \log \pi_{\theta_{old}}(a_t|s_t)\right)$, PPO maximise :
$$L^{CLIP}(\theta) = \mathbb{E}_t\left[\min\left(\rho_t(\theta)\hat{A}_t,\ \text{clip}(\rho_t(\theta), 1-\epsilon, 1+\epsilon)\hat{A}_t\right)\right]$$
Le clipping supprime l'intérêt de pousser $\rho_t$ au-delà de $[1-\epsilon, 1+\epsilon]$ : la nouvelle politique reste proche de l'ancienne.

**Loss totale** (à minimiser) :
$$L = -L^{CLIP} + c_V \, \mathbb{E}_t\left[(V_\phi(s_t) - R_t)^2\right] - c_H \, \mathbb{E}_t\left[H(\pi_\theta(\cdot|s_t))\right]$$
où le dernier terme est un bonus d'entropie qui encourage l'exploration.

## B) Rollout buffer

Ce buffer stocke les transitions du rollout en cours, avec la log-probabilité des actions jouées. Il est vidé après chaque mise à jour.

In [ ]:
class RolloutBuffer:
    def __init__(self):
        self.clear()

    def clear(self):
        self.obs = []
        self.actions = []
        self.log_probs = []  # log pi_old(a|s) of the played actions
        self.rewards = []
        self.next_obs = []
        self.terminateds = []
        self.dones = []  # terminated or truncated: end of the episode

    def append(self, obs, action, log_prob, reward, next_obs, terminated, done):
        self.obs.append(obs)
        self.actions.append(action)
        self.log_probs.append(log_prob)
        self.rewards.append(reward)
        self.next_obs.append(next_obs)
        self.terminateds.append(terminated)
        self.dones.append(done)

    def get(self, device):
        """Return the stored transitions as tensors, each of dim = (n_steps, ...)."""
        data = (self.obs, self.actions, self.log_probs, self.rewards, self.next_obs, self.terminateds, self.dones)
        return [torch.tensor(np.array(x), dtype=torch.float32, device=device) for x in data]

    def __len__(self):
        return len(self.obs)

## C) Réseaux de l'acteur et du critique

Deux petits réseaux entièrement connectés :
- l'**acteur** $\pi_\theta$ renvoie un logit par action, d'où la distribution `Categorical(logits=...)` sur les actions ;
- le **critique** $V_\phi$ renvoie une valeur par état.

In [ ]:
class PolicyNetwork(nn.Module):
    def __init__(self, obs_dim=4, n_actions=2, hidden_dim=64):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, n_actions)

    def forward(self, x):
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        return self.head(x)  # logits, dim = (batch_size, n_actions)


class ValueNetwork(nn.Module):
    def __init__(self, obs_dim=4, hidden_dim=64):
        super(ValueNetwork, self).__init__()
        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        return self.head(x).squeeze(-1)  # values, dim = (batch_size,)

Exemple : distribution de l'acteur (non entraîné) sur une observation, et action échantillonnée.

In [ ]:
obs, _ = env_cartpole.reset()
policy_network = PolicyNetwork()
dist = Categorical(logits=policy_network(torch.tensor(obs, dtype=torch.float32).unsqueeze(0)))
action = dist.sample()
print(f"Probabilités des actions : {dist.probs}")
print(f"Action échantillonnée : {action.item()}, log-probabilité : {dist.log_prob(action).item():.3f}")

## D) Agent PPO

À vous d'implémenter :
- `act` : échantillonner une action selon $\pi_\theta(\cdot|s)$ et mémoriser sa log-probabilité (elle sera stockée dans le buffer par `learn`, appelée juste après `act` par la boucle d'entraînement) ;
- `compute_advantages` : le calcul de $\hat{A}_t$ (GAE) et des cibles $R_t$ ;
- dans `update` : le ratio $\rho_t$, la loss de l'acteur, la loss du critique et la loss totale.

In [ ]:
class PPOAgent(AgentRL):

    def __init__(
        self,
        env : gym.Env,
        policy_network : nn.Module,
        value_network : nn.Module,
        n_steps : int,
        n_epochs : int,
        batch_size : int,
        learning_rate : float,
        gamma : float,
        gae_lambda : float,
        clip_epsilon : float,
        value_coef : float = 0.5,
        entropy_coef : float = 0.01,
        max_grad_norm : float = 0.5,
        device : str = "cpu",
    ):
        super().__init__(env)

        # Hyperparameters
        self.n_steps = n_steps  # number of steps collected between two updates
        self.n_epochs = n_epochs  # number of passes over the rollout at each update
        self.batch_size = batch_size  # minibatch size for the gradient steps
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.clip_epsilon = clip_epsilon
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm  # gradient clipping, for stability

        # Internal variables
        self.policy_network = policy_network.to(device)
        self.value_network = value_network.to(device)
        self.parameters = list(self.policy_network.parameters()) + list(self.value_network.parameters())
        self.optimizer = torch.optim.Adam(self.parameters, lr=learning_rate)
        self.buffer = RolloutBuffer()
        self.device = device
        self.last_log_prob = None  # log pi(a|s) of the action returned by the last call to act()

    def act(self, obs):
        with torch.no_grad():
            # ---- <your code here> ----
            action = ...
            self.last_log_prob = ...  # log-probability of the sampled action (float)
            # --------------------------
        return action

    def learn(self, obs, action, reward, next_obs, terminated, truncated):
        # Store the transition, then update the agent once the rollout is complete
        self.buffer.append(obs, action, self.last_log_prob, reward, next_obs, terminated, terminated or truncated)
        if len(self.buffer) >= self.n_steps:
            self.update()
            self.buffer.clear()  # on-policy: the data of the old policy is discarded

    def compute_advantages(self, rewards, values, next_values, terminateds, dones):
        """GAE advantages and value targets. All inputs and outputs have dim = (n_steps,)."""
        advantages = torch.zeros_like(rewards)
        # ---- <your code here> ----
        # Compute advantages[t] for t from the end of the rollout to the beginning
        ...
        returns = ...
        # --------------------------
        return advantages, returns

    def update(self):
        obs, actions, old_log_probs, rewards, next_obs, terminateds, dones = self.buffer.get(self.device)

        # Advantages and value targets, computed once with the networks before the update
        with torch.no_grad():
            values = self.value_network(obs)
            next_values = self.value_network(next_obs)
            advantages, returns = self.compute_advantages(rewards, values, next_values, terminateds, dones)

        for epoch in range(self.n_epochs):
            for indices in torch.randperm(len(obs)).split(self.batch_size):
                dist = Categorical(logits=self.policy_network(obs[indices]))
                log_probs = dist.log_prob(actions[indices].long())  # dim = (batch_size,)
                batch_advantages = advantages[indices]
                batch_advantages = (batch_advantages - batch_advantages.mean()) / (batch_advantages.std() + 1e-8)

                # ---- <your code here> ----
                ratio = ...
                policy_loss = ...
                value_loss = ...
                entropy = ...
                loss = ...
                # --------------------------

                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.parameters, self.max_grad_norm)
                self.optimizer.step()

Vérifions `compute_advantages` sur un petit exemple : avec $\lambda = 1$ et un critique qui prédit toujours 0, l'avantage est simplement le retour actualisé. Pour 3 récompenses de 1 avec $\gamma = 0.5$ et un épisode qui se termine au dernier pas : $\hat{A} = [1 + 0.5 + 0.25,\ 1 + 0.5,\ 1] = [1.75,\ 1.5,\ 1]$.

In [ ]:
test_agent = PPOAgent(env_cartpole, PolicyNetwork(), ValueNetwork(), n_steps=3, n_epochs=1, batch_size=3,
                      learning_rate=1e-3, gamma=0.5, gae_lambda=1.0, clip_epsilon=0.2)
zeros = torch.zeros(3)
advantages, returns = test_agent.compute_advantages(
    rewards=torch.ones(3), values=zeros, next_values=zeros,
    terminateds=torch.tensor([0., 0., 1.]), dones=torch.tensor([0., 0., 1.]),
)
print(f"Avantages : {advantages}")
assert torch.allclose(advantages, torch.tensor([1.75, 1.5, 1.0])), "compute_advantages est incorrect"
print("compute_advantages est correct !")

## E) Entraînement

C'est parti pour l'entraînement de notre agent PPO !

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

agent = PPOAgent(
    env = env_cartpole,
    policy_network = PolicyNetwork(
        obs_dim=env_cartpole.observation_space.shape[0],
        n_actions=env_cartpole.action_space.n,
    ),
    value_network = ValueNetwork(obs_dim=env_cartpole.observation_space.shape[0]),
    n_steps = 512,
    n_epochs = 10,
    batch_size = 64,
    learning_rate = 0.01,
    gamma = 0.99,
    gae_lambda = 0.95,
    clip_epsilon = 0.2,
    device = device,
)

# Train the agent
training_results = train_agent_n_episodes(env_cartpole, agent, n=300, verbose_freq=10)

# Plot training curves
plot_results(training_results)

# Evaluate the agent
eval_results = eval_agent_n_episodes(env_cartpole, agent, n=10)
print(f"Total reward: {eval_results['total_reward']}, mean = {np.mean(eval_results['total_reward'])}")
print(f"Total steps: {eval_results['steps']}, mean = {np.mean(eval_results['steps'])}")
print(f"Animation (last episode):")
eval_results['animation'][-1]

Remarques :
- Comparez la courbe d'apprentissage avec celle de DQN : celle de PPO est en général beaucoup plus régulière.
- L'agent n'apprend qu'une fois par rollout de `n_steps` pas : au début, les épisodes sont courts et il faut de nombreux épisodes avant la première mise à jour.
- Pour aller plus loin, observez l'effet des hyperparamètres : `n_epochs` (nombre de passes sur un même rollout), `clip_epsilon` (que se passe-t-il avec une valeur très grande, c'est-à-dire sans clipping ?), `entropy_coef`, `n_steps`.